In [1]:
# ==========================================================
# 🦐 AI-Based Whiteleg Shrimp Condition Detection (Image + Video)
# ==========================================================

# Step 1: Install dependencies
!pip install gradio torch torchvision pillow opencv-python --quiet

# Step 2: Import libraries
import torch
from torchvision import models, transforms
from PIL import Image
import gradio as gr
import torch.nn.functional as F
import cv2
import numpy as np
import tempfile

# Step 3: Load pretrained base model (ResNet18)
model = models.resnet18(weights='IMAGENET1K_V1')

# Change the final layer to match shrimp conditions (8 classes)
num_ftrs = model.fc.in_features
model.fc = torch.nn.Linear(num_ftrs, 8)

# Load trained weights if available
try:
    model.load_state_dict(torch.load("shrimp_model.pth", map_location=torch.device('cpu')))
    print("✅ Loaded trained Whiteleg Shrimp model.")
except:
    print("⚠️ No shrimp model found — using pretrained ResNet18 base (demo mode).")

model.eval()

# Step 4: Define image preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Step 5: Define shrimp condition classes
classes = [
    "Active Shrimp (Healthy)",
    "Stressed Shrimp",
    "Unusual Movement / Clustering",
    "Dead or Inactive Shrimp",
    "Poor Water Quality",
    "Feeding Area Overcrowded",
    "Possible Bacterial Infection",
    "Abnormal Swimming Pattern"
]

# Step 6: Prediction for a single image
def predict_image(image):
    img = transform(image).unsqueeze(0)

    with torch.no_grad():
        outputs = model(img)
        probs = F.softmax(outputs, dim=1)

    top5_prob, top5_catid = torch.topk(probs, 5)
    preds = {classes[top5_catid[0][i]]: float(top5_prob[0][i]) for i in range(5)}
    return preds

# Step 7: Prediction for video
def predict_video(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return {"Error": "Could not open video."}

    frame_count = 0
    predictions_accum = torch.zeros(len(classes))

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Sample every 10th frame for faster analysis
        if frame_count % 10 == 0:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = Image.fromarray(frame_rgb)
            img_t = transform(image).unsqueeze(0)
            with torch.no_grad():
                outputs = model(img_t)
                probs = F.softmax(outputs, dim=1)[0]
            predictions_accum += probs

        frame_count += 1

    cap.release()

    # Average prediction across frames
    predictions_accum /= max(1, (frame_count // 10))
    top5_prob, top5_catid = torch.topk(predictions_accum, 5)
    preds = {classes[top5_catid[i]]: float(top5_prob[i]) for i in range(5)}
    return preds

# Step 8: Unified handler for both image and video
def detect_shrimp_condition(input_media):
    if isinstance(input_media, str) and input_media.endswith((".mp4", ".avi", ".mov", ".mkv")):
        return predict_video(input_media)
    else:
        return predict_image(input_media)

# Step 9: Build Gradio Interface
interface = gr.Interface(
    fn=detect_shrimp_condition,
    inputs=[
        gr.File(label="Upload Image or Video (MP4, AVI, MOV)")
    ],
    outputs=gr.Label(num_top_classes=5, label="Predicted Shrimp Conditions"),
    title="🦐 Whiteleg Shrimp Condition Detection (Image + Video)",
    description=(
        "Upload an image or video of Whiteleg Shrimp (Litopenaeus vannamei). "
        "The AI model predicts health, stress, and abnormal conditions automatically."
    ),
    theme="soft"
)

# Step 10: Launch app
interface.launch(share=True)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 145MB/s]


⚠️ No shrimp model found — using pretrained ResNet18 base (demo mode).
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cd6b5a6ba393b19d25.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
